# Examen Fer León
Parcial 2

In [302]:
from collections import Counter
from itertools import product

import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth
from pyod.models.iforest import IForest
from pyod.models.lof import LOF
from sklearn.preprocessing import StandardScaler

from semestre_vii.mineria_de_datos import DATA_DIR

In [303]:
# cONFIGURACIONES DEL PROFE
soporte = 0.10
contaminacion = 0.007


pd.set_option("display.max_colwidth", None)

# Archivo mineria: compras del mismo ticket

El equipo de ventas busca optimizar el empaquetado de productos en el checkout. En este módulo no importa qué compró el cliente hace tres semanas; interesa saber qué artículos se compran juntos dentro del mismo ticket.

In [304]:
mineria = pd.read_csv(DATA_DIR / "mineria.csv")

cestas = pd.crosstab(mineria["Transaccion_ID"], mineria["Producto"]).gt(0)

In [305]:
frecuentes = fpgrowth(cestas, min_support=soporte, use_colnames=True)
reglas = association_rules(frecuentes, metric="lift", min_threshold=1.2)

P1.1: ¿Cuáles son las 3 reglas de asociación con mayor Lift y qué productos las componen?

In [306]:
reglas = reglas[["antecedents", "consequents", "support", "confidence", "lift"]]
reglas.columns = ["Antecedente", "Consecuente", "Soporte", "Confianza", "Lift"]

In [307]:
columnas = ["Antecedente", "Consecuente"]

for columna in columnas:
    reglas[columna] = reglas[columna].apply(lambda productos: ", ".join(sorted(productos)))

reglas = reglas.sort_values(["Lift", "Antecedente", "Consecuente"], ascending=[False, True, True])
reglas.head(3)

,Antecedente,Consecuente,Soporte,Confianza,Lift
0,Queso Curado,Vino Tinto,0.129043,1.0,7.749369
1,Vino Tinto,Queso Curado,0.129043,1.0,7.749369
17,Cafe en Grano,Cafetera Prensa,0.166160,1.0,6.018289


P1.2: Explica la diferencia práctica entre la Confianza y el Lift en la regla más fuerte encontrada.

In [308]:
fuerte = reglas.iloc[0]
antecedente = fuerte["Antecedente"]
consecuente = fuerte["Consecuente"]
confianza = fuerte["Confianza"]
lift = fuerte["Lift"]

print(f"{antecedente} -> {consecuente}")
print(f"Confianza {confianza:.0%}: tickets con X que también contienen Y.")
print(f"Lift {lift:.2f}: veces que supera la frecuencia general de Y.")


print(
"""
La diferencia entre confianza y el lift es que la confianza mide la probabilidad y el lift mide la fuerza  apartir
de la frecuenta de la asociación
"""
)


Queso Curado -> Vino Tinto
Confianza 100%: tickets con X que también contienen Y.
Lift 7.75: veces que supera la frecuencia general de Y.

La diferencia entre confianza y el lift es que la confianza mide la probabilidad y el lift mide la fuerza  apartir
de la frecuenta de la asociación



# Archivo mineria: compras en visitas posteriores

El equipo de marketing por correo electrónico desea saber: "Si un cliente compra el producto X hoy, ¿qué producto Y comprará en su siguiente visita y cuántos días debemos esperar para enviarle la campaña promocional?"

In [309]:
mineria["Fecha_Hora"] = pd.to_datetime(mineria["Fecha_Hora"])

agrupadores = ["Cliente_ID", "Fecha_Hora", "Transaccion_ID"]

visitas = mineria.groupby(
    agrupadores
)["Producto"].agg(list).reset_index()
clientes = visitas.groupby("Cliente_ID")

In [310]:
conteos = Counter()
for cliente, compras in clientes:
    cestas = compras["Producto"].tolist()
    cadenas = set()
    for largo in [2, 3]:
        for inicio in range(len(cestas) - largo + 1):
            cadenas.update(product(*cestas[inicio:inicio + largo]))
    conteos.update(cadenas)

In [311]:
columnas = ["Cadena", "Clientes"]

patrones = pd.DataFrame(conteos.items(), columns=columnas)
patrones["Soporte"] = patrones["Clientes"] / len(clientes)
patrones = patrones[patrones["Soporte"] >= soporte].sort_values("Clientes", ascending=False)

P2.1: ¿Cuáles son las dos cadenas secuenciales (una de 2 elementos y otra de 3 elementos) que superan el soporte mínimo?

In [312]:
dos = patrones[patrones["Cadena"].str.len() == 2].head(1)
tres = patrones[patrones["Cadena"].str.len() == 3].head(1)
pd.concat([dos, tres], ignore_index=True)

,Cadena,Clientes,Soporte
0,"(Smartphone Alpha, Funda Silicona)",346,0.231438
1,"(Smartphone Alpha, Funda Silicona, Mica Privacidad)",265,0.177258


P2.2: ¿Cuál es la ventana de días óptima calculada entre el primer producto y el segundo en la cadena de 3 pasos?

In [313]:
cadena = tres.iloc[0]["Cadena"]
dias = []

In [314]:
for cliente, compras in clientes:
    cestas = compras["Producto"].tolist()
    fechas = compras["Fecha_Hora"].tolist()
    for inicio in range(len(cestas) - 2):
        primero = cadena[0] in cestas[inicio]
        segundo = cadena[1] in cestas[inicio + 1]
        tercero = cadena[2] in cestas[inicio + 2]
        if primero and segundo and tercero:
            demora = fechas[inicio + 1] - fechas[inicio]
            dias.append(demora.total_seconds() / 86400)
            break

In [315]:
espera = pd.Series(dias).median()
total_dias = len(dias)

print(f"Cadena completa encontrada en {total_dias} clientes.")
print(f"Mediana: {espera:.2f} días. Espera sugerida: {espera:.0f} días.")

Cadena completa encontrada en 265 clientes.
Mediana: 4.04 días. Espera sugerida: 4 días.


# Archivo tarjetas: anomalías globales

Existen atacantes que utilizan bots para realizar compras con tarjetas robadas. Realizan un checkout express en menos de 1 minuto, con pocas sesiones pero montos altos. Ninguna de estas variables aisladas está fuera de los rangos globales de la plataforma, pero su correlación conjunta es anómala. Usa una contaminación de 0.7% y random_state = 42.

In [316]:
tarjetas = pd.read_csv(DATA_DIR / "Tarjetas.csv")
variables = [
    "Gasto_Mensual_USD",
    "Frecuencia_Sesiones_Semana",
    "Minutos_Promedio_Sesion",
    "Acciones_Por_Sesion",
]
datos = tarjetas[variables].values

In [317]:
bosque = IForest(n_estimators=100, contamination=contaminacion, random_state=42)
bosque.fit(datos)
tarjetas["iforest"] = bosque.labels_

P3.1: ¿Cuántas anomalías del tipo Anomalia_Global_BotGasto logró capturar el modelo?

In [318]:
bots = tarjetas["Etiqueta_Control_Examen"] == "Anomalia_Global_BotGasto"
total_bots = bots.sum()
detecciones = tarjetas.loc[bots, 'iforest'].sum()

print(f"iForest detectó {detecciones} de {total_bots} bots.")

iForest detectó 25 de 25 bots.


P3.2: ¿Por qué __________ captura casi la totalidad de Anomalia_Global_BotGasto, pero falla por completo en detectar Anomalia_Local_AbusoGratuito?

In [319]:
abuso = tarjetas["Etiqueta_Control_Examen"] == "Anomalia_Local_AbusoGratuito"
total_abuso = abuso.sum()
detecciones_abuso = tarjetas.loc[abuso, 'iforest'].sum()

print(f"iForest detectó {detecciones_abuso} de {total_abuso} abusos locales.")


print(
"""
Falla por que iForest es para anomalias que ocurren en todo el conjunto de alguna manera
Pero LOF es para las locales, segun yo
"""
)

iForest detectó 0 de 25 abusos locales.

Falla por que iForest es para anomalias que ocurren en todo el conjunto de alguna manera
Pero LOF es para las locales, segun yo



# Archivo tarjetas: anomalías locales

Dentro del mismo dataset coexisten dos comunidades: Comunidad_Gratuita (un grupo sumamente denso y homogéneo) y Empresarial_Pro (un grupo legítimo pero disperso). Un grupo de usuarios de la comunidad gratuita está abusando de las cuotas del sistema mediante micro-scripts de scraping. **Estandarización obligatoria:** aplica StandardScaler sobre las 4 variables numéricas.

In [320]:
escalador = StandardScaler()
escalados = escalador.fit_transform(datos)

vecinos = LOF(n_neighbors=50, contamination=contaminacion)
vecinos.fit(escalados)
tarjetas["lof"] = vecinos.labels_

P4.1: ¿Cuántas anomalías del tipo Anomalia_Local_AbusoGratuito logró aislar?

In [321]:
detecciones_lof = tarjetas.loc[abuso, 'lof'].sum()


print(f"LOF detectó {detecciones_lof} de {total_abuso} abusos locales.")

LOF detectó 25 de 25 abusos locales.


P4.2: Revisa los falsos positivos generados en el segmento Empresarial_Pro. ¿Cuál de los dos algoritmos (iForest o LOF) castigó menos a los usuarios empresariales legítimos y por qué?

In [322]:
normales = tarjetas["Etiqueta_Control_Examen"] == "Normal"
empresa = tarjetas["Segmento_Plan"] == "Empresarial_Pro"
falsos = tarjetas.loc[normales & empresa, ["iforest", "lof"]].sum().rename("Falsos positivos")

falsos




iforest    31
lof         1
Name: Falsos positivos, dtype: int64

In [323]:
print(
"""
IfOREST logro detectar anomalías en todo el conjunto de datos pero
no podria decir yo que LOF es mejor pero al menos fue el que menos dio falsos positivos
"""
)


IfOREST logro detectar anomalías en todo el conjunto de datos pero
no podria decir yo que LOF es mejor pero al menos fue el que menos dio falsos positivos

